In [56]:
import pandas as pd
from scipy import stats
import scikit_posthocs as sp

In [57]:
df = pd.read_csv("events.csv")

# Hypothesis Testing

### Conversion Time Analysis
Here it was evaluated whether conversion speed (time from session start to purchase) differs significantly across four key features at a 95% confidence level:

- Purchase Types (First-time vs. Repeat)

- Device Categories

- Traffic Sources

- Weekdays

### Methodology

Conversion time data is heavily skewed by a few unusually long sessions. Standard averages would distort the results, so non-parametric rank-based tests were used:

- Mann-Whitney U Test (Purchase Types & Device Categories): Used for two-group comparisons. It relies on medians rather than averages, making it resistant to extreme outliers and unequal group sizes.

- Kruskal-Wallis & Dunn's Test (Traffic Sources & Weekdays): Used for three or more groups. An initial Kruskal-Wallis test checks if any group stands out. When it does, a post-hoc Dunn's test pinpoints the exact differing pairs without raising false positives.

- Bonferroni Correction: When testing multiple pairs simultaneously (like 21 day-of-week combinations), false positive risks compound. Bonferroni adjustment was applied to penalize the p-values for every extra comparison, ensuring our overall confidence level strictly stayed at 95%.

### Key Findings

- First vs. Repeat Purchases: Statistical significance confirmed ($p < 0.05$). Repeat buyers convert faster than first-time buyers.
- Weekday Differences: Significant differences emerged across days ($p < 0.05$), driven by Wednesday (slowest conversion times) versus Sunday (fastest conversion times).

### Purchase Types

In [58]:
conversion_a = df[df["purchase_type"] == "first_buy"]["conversion_seconds"]
conversion_b = df[df["purchase_type"] == "repeat_buy"]["conversion_seconds"]
stat, p_value = stats.mannwhitneyu(
    conversion_a, conversion_b, alternative="two-sided"
)
print(f"P Value: {p_value:.5f}\n")
if p_value < 0.05:
    print(
        "Statistically significant difference detected (95% confidence)."
    )
else:
    print("No statistically significant difference detected.")

P Value: 0.00000

Statistically significant difference detected (95% confidence).


### Categories

In [59]:
conversion_a = df[df["category"] == "desktop"]["conversion_seconds"]
conversion_b = df[df["category"] == "mobile"]["conversion_seconds"]
stat, p_value = stats.mannwhitneyu(
    conversion_a, conversion_b, alternative="two-sided"
)
print(f"P Value: {p_value:.5f}\n")
if p_value < 0.05:
    print(
        "Statistically significant difference detected (95% confidence)."
    )
else:
    print("No statistically significant difference detected.")

P Value: 0.67789

No statistically significant difference detected.


### Traffic Source

In [60]:
groups = []
for group_name, group_data in df.groupby("traffic_source"):
    time_values = group_data["conversion_seconds"].values
    groups.append(time_values)
kw_stat, kw_pvalue = stats.kruskal(*groups)

if kw_pvalue < 0.05:
    dunn_results = sp.posthoc_dunn(
        df,
        val_col="conversion_seconds",
        group_col="traffic_source",
        p_adjust="bonferroni",
    )
    results = dunn_results.unstack().reset_index()
    results.columns = ["Day A", "Day B", "p_value"]
    significant_results = results[
        (results["p_value"] < 0.05) & (results["Day A"] != results["Day B"])
        ]
    significant_results = significant_results[
        significant_results["Day A"] < significant_results["Day B"]
        ]
    print("Statistically Significant Results (p < 0.05):")
    print(significant_results.to_string(index=False))
else:
    print("No statistically significant differences detected across groups.")

No statistically significant differences detected across groups.


### Weekdays

In [61]:
df["session_date"] = pd.to_datetime(df["session_date"])
df["weekday_name"] = df["session_date"].dt.day_name()

In [62]:
groups = []
for group_name, group_data in df.groupby("weekday_name"):
    time_values = group_data["conversion_seconds"].values
    groups.append(time_values)
kw_stat, kw_pvalue = stats.kruskal(*groups)

if kw_pvalue < 0.05:
    dunn_results = sp.posthoc_dunn(
        df,
        val_col="conversion_seconds",
        group_col="weekday_name",
        p_adjust="bonferroni",
    )
    results = dunn_results.unstack().reset_index()
    results.columns = ["Day A", "Day B", "p_value"]
    significant_results = results[
        (results["p_value"] < 0.05) & (results["Day A"] != results["Day B"])
        ]
    significant_results = significant_results[
        significant_results["Day A"] < significant_results["Day B"]
        ]
    print("Statistically Significant Results (p < 0.05):")
    print(significant_results.to_string(index=False))
else:
    print("\nNo statistically significant differences detected across groups.")

Statistically Significant Results (p < 0.05):
 Day A     Day B  p_value
Sunday Wednesday 0.001037
